<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 14</h2>
<h2>MD5, SHA-1, SHA-2</h2>
<h3 style="color: #8B4513;">Heş funksiyalarının təkamülü və təhlükəsizlik</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: Heş funksiyalarının təkamülü (10 dəq)](#xatırlatma-heş-funksiyalarının-təkamülü-10-dəq)
5. [MD5 heş funksiyası (15 dəq)](#md5-heş-funksiyası-15-dəq)
6. [SHA-1 heş funksiyası (15 dəq)](#sha-1-heş-funksiyası-15-dəq)
7. [SHA-2 ailəsi (20 dəq)](#sha-2-ailəsi-20-dəq)
8. [Müasir tətbiqlərdə heş funksiyaları (15 dəq)](#müasir-tətbiqlərdə-heş-funksiyaları-15-dəq)
9. [İnteqrasiya edilmiş tətbiq (15 dəq)](#inteqrasiya-edilmiş-tətbiq-15-dəq)
10. [Ev tapşırığı](#ev-tapşırığı)
11. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
12. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ MD5, SHA-1 və SHA-2 heş funksiyalarının tarixini və xüsusiyyətlərini izah edə biləcəksiniz  
✅ MD5 və SHA-1-ə qarşı aparılmış hücumların mahiyyətini başa düşəcəksiniz  
✅ Python-da müxtəlif heş funksiyalarından istifadə edə biləcəksiniz  
✅ SHA-256 və SHA-512-nin performansını müqayisə edə biləcəksiniz  
✅ Müasir tətbiqlər üçün uyğun heş funksiyasını seçə biləcəksiniz  
✅ Köhnə heş funksiyalarından yeni sistemlərə keçid strategiyalarını biləcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | Quraşdırma | İstifadə sahəsi |
|-----------|------------|-----------------|
| hashlib | Python standart kitabxanası | MD5, SHA-1, SHA-2, SHA-3 |
| time | Python standart kitabxanası | Performans ölçmə |
| os, secrets | Python standart kitabxanası | Təsadüfi duz (salt) generasiyası |
| matplotlib | `!pip install matplotlib` | Qrafiklər çəkmək üçün |
| numpy | `!pip install numpy` | Rəqəmsal hesablamalar |

> 💡 **Qeyd:** hashlib modulu Python-un standart kitabxanasına daxildir. Matplotlib və numpy isə isteğe bağlıdır.

In [ ]:
# Lazımi kitabxanaların yoxlanılması və yalnız çatışmayanların quraşdırılması

import importlib.util
import subprocess
import sys

required_packages = ["matplotlib", "numpy"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    print("✅ Çatışmayan kitabxanalar quraşdırıldı:", ", ".join(missing_packages))
else:
    print("✅ Lazımi kitabxanalar artıq quraşdırılıb")


## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import sys
import hashlib
import time
import os
import secrets
import math
import random
from pathlib import Path
from collections import Counter

# Əlavə kitabxanalar (isteğe bağlı)
try:
    import numpy as np
    import matplotlib.pyplot as plt
    print("✅ numpy və matplotlib yüklü - qrafiklər çəkilə bilər")
    PLT_AVAILABLE = True
except ImportError:
    print("⚠️ Bəzi kitabxanalar yoxdur, sadəcə əsas funksiyalar işləyəcək")
    PLT_AVAILABLE = False

print(f"\n🐍 Python versiyası: {sys.version}")

# Mövcud heş alqoritmlərini göstər
available_algorithms = sorted(hashlib.algorithms_available)
print("\n📋 Mövcud heş alqoritmləri:")
for algorithm in available_algorithms[:10]:
    print(f"  - {algorithm}")
print(f"  ... və daha {max(0, len(available_algorithms)-10)} alqoritm")


### 3.2 İşçi qovluğun yaradılması

In [ ]:
import os
from pathlib import Path

# İşçi qovluğu yaradaq (hüceyrə təkrar icra olunsa da iç-içə qovluq yaratmasın)
current_dir = Path.cwd().resolve()

if current_dir.parts[-2:] == ("crypto_workshop", "lecture14"):
    workspace = current_dir
else:
    workspace = current_dir / "crypto_workshop" / "lecture14"

workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)

print(f"📁 İşçi qovluq: {workspace}")
print(f"📂 Qovluğun məzmunu: {sorted(os.listdir('.'))}")


### 3.3 Yardımçı funksiyalar

In [ ]:
def to_bytes(data):
    """
    Girişi bytes formasına çevir.
    """
    if isinstance(data, bytes):
        return data
    if isinstance(data, str):
        return data.encode("utf-8")
    raise TypeError("Giriş str və ya bytes olmalıdır")


def double_sha256(data):
    """
    Bitcoin-də geniş istifadə olunan ikiqat SHA-256 (SHA-256d).
    """
    data = to_bytes(data)
    return hashlib.sha256(hashlib.sha256(data).digest()).hexdigest()


def mean_hash_time(hash_constructor, data, repeats):
    """
    Verilmiş heş funksiyası üçün orta işləmə vaxtını (millisaniyə) hesabla.
    """
    start = time.perf_counter()
    for _ in range(repeats):
        hash_constructor(data).digest()
    elapsed = time.perf_counter() - start
    return (elapsed * 1000) / repeats


## 📖 Xatırlatma: Heş funksiyalarının təkamülü (10 dəq)

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 Heş funksiyalarının inkişafı</h4>
<ul>
    <li><b>1991</b>: MD5 (Ron Rivest) - 128-bit, geniş yayıldı</li>
    <li><b>1995</b>: SHA-1 (NSA) - 160-bit, MD5-i əvəz etdi</li>
    <li><b>2001</b>: SHA-2 (NSA/NIST) - 224/256/384/512-bit, daha güclü və daha uzun heş ailəsi</li>
    <li><b>2004</b>: MD5 toqquşmaları tapıldı (Wang)</li>
    <li><b>2017</b>: SHA-1 toqquşması (SHAttered)</li>
    <li><b>2015+</b>: SHA-3 (Keccak) - süngər quruluşu</li>
</ul>
</div>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4; margin-top: 10px;">
<h4>🔑 Heş funksiyalarının təhlükəsizlik səviyyəsi</h4>
<ul>
    <li><b>MD5</b> (128-bit): Tamamilə sındırılmış, istifadə edilməməlidir</li>
    <li><b>SHA-1</b> (160-bit): Etibarsız, praktik toqquşmalar mövcud</li>
    <li><b>SHA-256</b> (256-bit): Təhlükəsiz, 128-bit təhlükəsizlik səviyyəsi</li>
    <li><b>SHA-512</b> (512-bit): Təhlükəsiz, 256-bit təhlükəsizlik səviyyəsi</li>
</ul>
</div>


## 🔐 MD5 heş funksiyası (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>📌 MD5 xüsusiyyətləri</h4>
<ul>
    <li><b>Müəllif:</b> Ron Rivest (1991)</li>
    <li><b>Çıxış uzunluğu:</b> 128-bit (16 bayt)</li>
    <li><b>Blok ölçüsü:</b> 512-bit (64 bayt)</li>
    <li><b>Raund sayı:</b> 64</li>
    <li><b>Status:</b> Tamamilə sındırılmış (2004)</li>
</ul>
</div>

In [ ]:
def md5_demo():
    """
    MD5 heş funksiyasının nümayişi
    """
    print("=" * 70)
    print("🔐 MD5 HEŞ FUNKSİYASI")
    print("=" * 70)

    # Müxtəlif mətnlərin MD5 heşi
    texts = [
        "Salam, dunya!",
        "Salam, dunya?",
        "Kriptoqrafiya kursu",
        "Kriptoqrafiya kursu."  # bir nöqtə fərqi
    ]

    for text in texts:
        md5_hash = hashlib.md5(text.encode()).hexdigest()
        print(f"\n📨 Mətn: {text}")
        print(f"🔹 MD5: {md5_hash}")
        print(f"   Uzunluq: {len(md5_hash)} hex simvol = {len(md5_hash)*4} bit")

    return {text: hashlib.md5(text.encode()).hexdigest() for text in texts}

md5_hashes = md5_demo()

### 5.1 MD5 toqquşma nümunəsi

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ MD5 toqquşması</h4>
<p>2004-cü ildə Xiaoyun Wang və digərləri MD5-də praktik toqquşmalar göstərdilər. Bu gün hazır alətlər və dərc olunmuş nümunələr mövcuddur; buna görə MD5 kriptoqrafik təhlükəsizlik üçün uyğun deyil.</p>
</div>


In [ ]:
def md5_collision_demo():
    """
    MD5 üçün real toqquşma nümunəsi.
    İki fərqli 128-baytlıq mesajın MD5 heşi eynidir.
    """
    print("\n" + "-" * 70)
    print("💥 MD5 TOQQUŞMA NÜMUNƏSİ")
    print("-" * 70)

    # Klassik, dərc olunmuş MD5 toqquşma cütü
    hex1 = (
        "d131dd02c5e6eec4693d9a0698aff95c"
        "2fcab58712467eab4004583eb8fb7f89"
        "55ad340609f4b30283e488832571415a"
        "085125e8f7cdc99fd91dbdf280373c5b"
        "d8823e3156348f5bae6dacd436c919c6"
        "dd53e2b487da03fd02396306d248cda0"
        "e99f33420f577ee8ce54b67080a80d1e"
        "c69821bcb6a8839396f9652b6ff72a70"
    )

    hex2 = (
        "d131dd02c5e6eec4693d9a0698aff95c"
        "2fcab50712467eab4004583eb8fb7f89"
        "55ad340609f4b30283e4888325f1415a"
        "085125e8f7cdc99fd91dbd7280373c5b"
        "d8823e3156348f5bae6dacd436c919c6"
        "dd53e23487da03fd02396306d248cda0"
        "e99f33420f577ee8ce54b67080280d1e"
        "c69821bcb6a8839396f965ab6ff72a70"
    )

    bytes1 = bytes.fromhex(hex1)
    bytes2 = bytes.fromhex(hex2)

    hash1 = hashlib.md5(bytes1).hexdigest()
    hash2 = hashlib.md5(bytes2).hexdigest()

    print(f"Giriş 1 uzunluğu: {len(bytes1)} bayt")
    print(f"Giriş 2 uzunluğu: {len(bytes2)} bayt")
    print(f"Girişlər fərqlidir? {bytes1 != bytes2}")
    print(f"Giriş 1 (ilk 64 hex): {hex1[:64]}...")
    print(f"Giriş 2 (ilk 64 hex): {hex2[:64]}...")

    print(f"\nMD5(giriş 1): {hash1}")
    print(f"MD5(giriş 2): {hash2}")
    print(f"\n✅ Heşlər bərabərdir? {hash1 == hash2}")

    if hash1 == hash2 and bytes1 != bytes2:
        print("\n⚠️ Nəticə: fərqli iki mesaj eyni MD5 heşinə malikdir.")
        print("MD5 kriptoqrafik təhlükəsizlik üçün uyğun deyil.")
    else:
        print("\n❌ Toqquşma nümayişi alınmadı.")

    return {
        "messages_different": bytes1 != bytes2,
        "hash1": hash1,
        "hash2": hash2,
        "collision_verified": hash1 == hash2,
    }

md5_collision_info = md5_collision_demo()


### ✍️ Çalışma 1: MD5 (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Şəxsi heş:** Öz adınızın MD5 heşini hesablayın.

2. **Böyük/kiçik hərf:** Eyni mətnin böyük hərf və kiçik hərf variantlarının MD5 heşlərini müqayisə edin.

3. **Nəzəri mürəkkəblik:** MD5-in çıxış uzunluğu 128-bitdir. Ad günü paradoksuna əsasən, MD5-də toqquşma tapmaq üçün nəzəri mürəkkəblik nədir?

4. **Nəzəri sual:** Niyə MD5 bu gün təhlükəsiz hesab edilmir?

In [ ]:
# Çalışma 1 - Cavablar

print("📝 ÇALIŞMA 1 CAVABLARI")
print("=" * 80)

# 1. Şəxsi heş
print("\n1. ŞƏXSİ HEŞ:")
my_name = "Səməd Vəkilov"  # Öz adınızla əvəz edin
name_md5 = hashlib.md5(my_name.encode("utf-8")).hexdigest()
print(f"   Ad: {my_name}")
print(f"   MD5: {name_md5}")

# 2. Böyük/kiçik hərf müqayisəsi
print("\n2. BÖYÜK/KİÇİK HƏRF MÜQAYİSƏSİ:")
lower_text = "salam"
upper_text = "SALAM"
h_lower = hashlib.md5(lower_text.encode("utf-8")).hexdigest()
h_upper = hashlib.md5(upper_text.encode("utf-8")).hexdigest()
print(f"   'salam' MD5: {h_lower}")
print(f"   'SALAM' MD5: {h_upper}")
print(f"   Heşlər eynidir? {h_lower == h_upper}")

# 3. Nəzəri mürəkkəblik
print("\n3. NƏZƏRİ MÜRƏKKƏBLİK:")
print("   MD5 üçün toqquşma mürəkkəbliyi: 2^(128/2) = 2^64 ≈ 1.8 × 10^19")

# 4. MD5-in təhlükəsizlik problemi
print("\n4. MD5 TƏHLÜKƏSİZLİK PROBLEMİ:")
print("""
   MD5 bu gün təhlükəsiz hesab edilmir, çünki:
   • Praktik toqquşma nümunələri mövcuddur
   • Rəqəmsal imza və sertifikat kimi təhlükəsizlik-kritik işlər üçün yararsızdır
   • Yeni sistemlərdə SHA-256 və ya daha güclü alternativlər seçilməlidir
   • Sadə checksum üçün işləsə də, kriptoqrafik təhlükəsizlik təmin etmir
""")


## 🔐 SHA-1 heş funksiyası (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>📌 SHA-1 xüsusiyyətləri</h4>
<ul>
    <li><b>Müəllif:</b> NSA (1995)</li>
    <li><b>Çıxış uzunluğu:</b> 160-bit (20 bayt)</li>
    <li><b>Blok ölçüsü:</b> 512-bit (64 bayt)</li>
    <li><b>Raund sayı:</b> 80</li>
    <li><b>Status:</b> Etibarsız, praktik toqquşmalar mövcud (2017)</li>
</ul>
</div>

In [ ]:
def sha1_demo():
    """
    SHA-1 heş funksiyasının nümayişi
    """
    print("\n" + "=" * 70)
    print("🔐 SHA-1 HEŞ FUNKSİYASI")
    print("=" * 70)

    text = "Kriptoqrafiya kursu"
    sha1_hash = hashlib.sha1(text.encode("utf-8")).hexdigest()

    print(f"📨 Mətn: {text}")
    print(f"🔹 SHA-1: {sha1_hash}")
    print(f"   Uzunluq: {len(sha1_hash)} hex simvol = {len(sha1_hash)*4} bit")

    # MD5 ilə müqayisə
    md5_hash = hashlib.md5(text.encode("utf-8")).hexdigest()
    print(f"\n📊 MÜQAYİSƏ:")
    print(f"   MD5:   {md5_hash}")
    print(f"   SHA-1: {sha1_hash}")
    print(f"   SHA-1 MD5-dən 32-bit daha uzundur (160 vs 128)")

    return {"text": text, "MD5": md5_hash, "SHA-1": sha1_hash}

sha1_hashes = sha1_demo()


### 6.1 SHAttered hücumu (2017)

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 SHAttered hücumu</h4>
<p>2017-ci ildə Google və CWI Amsterdam tədqiqatçıları SHA-1 üçün ilk praktik toqquşmanı nümayiş etdirdilər. Hücumun mürəkkəbliyi $2^{63}$ əməliyyat idi (~110 il GPU vaxtı).</p>
</div>

In [ ]:
def shattered_demo():
    """
    SHAttered hücumu haqqında məlumat
    """
    print("\n" + "-" * 70)
    print("💥 SHATTERED HÜCUMU (SHA-1)")
    print("-" * 70)

    print("""
    SHAttered hücumu (2017):
    • İlk praktik SHA-1 toqquşması
    • Eyni SHA-1 heşinə malik iki fərqli PDF faylı yaradıldı
    • Mürəkkəblik: 2^63 əməliyyat səviyyəsində qiymətləndirilirdi
    • Təxmini xərc: böyük bulud hesablama resursları

    SHA-1 heşi: 38762cf7f55934b34d179ae6a4c80cadccbb7f0a

    Təsirlər:
    • Git versiya idarəetmə sistemi
    • SSL sertifikatları
    • Köhnə təhlükəsizlik sistemləri

    Qeyd:
    • Bu notebook hücumu hesablamır; yalnız tarixi faktı izah edir.
    """)

    return {
        "collision_hash": "38762cf7f55934b34d179ae6a4c80cadccbb7f0a",
        "year": 2017,
        "practical_collision": True,
    }

shattered_info = shattered_demo()


### ✍️ Çalışma 2: SHA-1 (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Şəxsi heş:** Öz adınızın SHA-1 heşini hesablayın və MD5 ilə müqayisə edin.

2. **Nəzəri mürəkkəblik:** SHA-1-in çıxış uzunluğu 160-bitdir. Toqquşma tapmaq üçün nəzəri mürəkkəblik nədir?

3. **SHAttered haqqında:** SHAttered hücumu nədir və niyə əhəmiyyətlidir?

4. **Git və SHA-1:** SHA-1 hələ də Git-də istifadə olunur. Bu, təhlükəsizlik riski yaradırmı?

In [ ]:
# Çalışma 2 - Cavablar

print("📝 ÇALIŞMA 2 CAVABLARI")
print("=" * 80)

# 1. Şəxsi heş
print("\n1. ŞƏXSİ HEŞ:")
my_name = "Səməd Vəkilov"
name_sha1 = hashlib.sha1(my_name.encode("utf-8")).hexdigest()
name_md5 = hashlib.md5(my_name.encode("utf-8")).hexdigest()
print(f"   Ad: {my_name}")
print(f"   MD5: {name_md5}")
print(f"   SHA-1: {name_sha1}")

# 2. Nəzəri mürəkkəblik
print("\n2. NƏZƏRİ MÜRƏKKƏBLİK:")
print("   SHA-1 üçün toqquşma mürəkkəbliyi: 2^(160/2) = 2^80 ≈ 1.2 × 10^24")

# 3. SHAttered haqqında
print("\n3. SHATTERED HÜCUMU:")
print("""
   SHAttered hücumu (2017):
   • SHA-1 üçün ilk praktik toqquşma
   • Mürəkkəblik nəzəri 2^80 həddindən xeyli aşağı praktik səviyyəyə endirildi
   • Əhəmiyyəti: SHA-1-in etibarsız olduğunu praktik olaraq göstərdi
   • Yeni sistemlər üçün SHA-1 artıq tövsiyə olunmur
""")

# 4. Git və SHA-1
print("\n4. GIT VƏ SHA-1:")
print("""
   Git hələ də SHA-1 istifadə edən obyektlərlə işləyə bilir, lakin bu ideal deyil:
   • Git obyektləri sadə fayl məzmunundan daha çox məlumatı heşləyir
   • Bu, hücumu çətinləşdirir, amma problemi tam aradan qaldırmır
   • Buna görə Git SHA-256 dəstəyi əlavə edib
   • Yeni dizaynlar üçün SHA-256 daha münasib seçimdir
""")


## 🔐 SHA-2 ailəsi (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>📌 SHA-2 ailəsi</h4>
<p>SHA-2 (Secure Hash Algorithm 2) — NSA tərəfindən hazırlanmış, 2001-ci ildə NIST tərəfindən standartlaşdırılmış heş funksiyaları ailəsidir.</p>
</div>

| Alqoritm | Çıxış uzunluğu | Blok ölçüsü | Raund sayı | Təhlükəsizlik |
|----------|----------------|-------------|------------|----------------|
| SHA-224 | 224-bit | 512-bit | 64 | 112-bit |
| SHA-256 | 256-bit | 512-bit | 64 | 128-bit |
| SHA-384 | 384-bit | 1024-bit | 80 | 192-bit |
| SHA-512 | 512-bit | 1024-bit | 80 | 256-bit |

In [ ]:
def sha2_family_demo():
    """
    SHA-2 ailəsinin nümayişi
    """
    print("\n" + "=" * 70)
    print("🔐 SHA-2 AİLƏSİ")
    print("=" * 70)

    text = "Kriptoqrafiya kursu 2024"
    data = text.encode("utf-8")

    print(f"📨 Mətn: {text}\n")

    # SHA-224
    h224 = hashlib.sha224(data).hexdigest()
    print(f"🔹 SHA-224 (224-bit): {h224}")

    # SHA-256
    h256 = hashlib.sha256(data).hexdigest()
    print(f"\n🔹 SHA-256 (256-bit): {h256}")

    # SHA-384
    h384 = hashlib.sha384(data).hexdigest()
    print(f"\n🔹 SHA-384 (384-bit): {h384[:64]}...")

    # SHA-512
    h512 = hashlib.sha512(data).hexdigest()
    print(f"\n🔹 SHA-512 (512-bit): {h512[:64]}...")

    return {
        "SHA-224": h224,
        "SHA-256": h256,
        "SHA-384": h384,
        "SHA-512": h512,
    }

sha2_hashes = sha2_family_demo()


### 7.1 SHA-256 vs SHA-512 performans müqayisəsi

In [ ]:
def performance_comparison():
    """
    SHA-256 və SHA-512-nin performans müqayisəsi
    """
    print("\n" + "-" * 70)
    print("⚡ PERFORMANS MÜQAYİSƏSİ")
    print("-" * 70)

    # (ölçü, təkrar sayı)
    cases = [
        (1 * 1024, 5000),
        (10 * 1024, 2000),
        (100 * 1024, 300),
        (1 * 1024 * 1024, 30),
    ]

    print(f"{'Ölçü':<10} {'Təkrar':<10} {'SHA-256 (ms)':<15} {'SHA-512 (ms)':<15} {'Nisbət':<10}")
    print("-" * 70)

    sizes_kb = []
    sha256_times = []
    sha512_times = []

    for size, repeats in cases:
        data = os.urandom(size)

        t256 = mean_hash_time(hashlib.sha256, data, repeats)
        t512 = mean_hash_time(hashlib.sha512, data, repeats)
        ratio = t512 / t256 if t256 else float("nan")

        sizes_kb.append(size / 1024)
        sha256_times.append(t256)
        sha512_times.append(t512)

        print(f"{size/1024:>6.1f} KB   {repeats:<10} {t256:<15.4f} {t512:<15.4f} {ratio:<10.2f}")

    if PLT_AVAILABLE:
        plt.figure(figsize=(10, 6))
        plt.plot(sizes_kb, sha256_times, marker='o', label='SHA-256')
        plt.plot(sizes_kb, sha512_times, marker='s', label='SHA-512')
        plt.xlabel('Məlumat ölçüsü (KB)')
        plt.ylabel('Orta vaxt (ms)')
        plt.title('SHA-256 vs SHA-512 Performans Müqayisəsi')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    faster = "SHA-256" if sum(sha256_times) < sum(sha512_times) else "SHA-512"
    print(f"\n📌 Bu sistemdə orta hesabla daha sürətli alqoritm: {faster}")
    print("📌 QEYD: nəticə platformadan, OpenSSL implementasiyasından və CPU arxitekturasından asılıdır.")

    return list(zip(sizes_kb, sha256_times, sha512_times))

performance_results = performance_comparison()


### 7.2 SHA-2-nin təhlükəsizliyi

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🛡️ SHA-2 təhlükəsizlik səviyyəsi</h4>
<ul>
    <li><b>Ön görüntü dayanıqlılığı:</b> $2^n$ (SHA-256 üçün $2^{256}$)</li>
    <li><b>Toqquşma dayanıqlılığı:</b> $2^{n/2}$ (SHA-256 üçün $2^{128}$)</li>
    <li><b>Uzunluq genişləndirmə:</b> SHA-256 və SHA-512 kimi tam vəziyyəti çıxışa verən Merkle–Damgård variantlarında həssas; SHA-224 və SHA-384 kimi truncasiya olunmuş variantlarda hücum birbaşa tətbiq olunmur.</li>
    <li><b>Praktik hücumlar:</b> Yoxdur</li>
</ul>
</div>

In [ ]:
def security_comparison():
    """
    Heş funksiyalarının təhlükəsizlik müqayisəsi
    """
    print("\n" + "-" * 70)
    print("📊 TƏHLÜKƏSİZLİK MÜQAYİSƏSİ")
    print("-" * 70)

    algorithms = [
        ("MD5", 128, "Sındırılmış", "Bəli (2004)"),
        ("SHA-1", 160, "Etibarsız", "Bəli (2017)"),
        ("SHA-256", 256, "Təhlükəsiz", "Xeyr"),
        ("SHA-512", 512, "Təhlükəsiz", "Xeyr")
    ]

    print(f"{'Alqoritm':<10} {'Çıxış':<10} {'Status':<15} {'Praktik toqquşma':<20}")
    print("-" * 60)

    for name, bits, status, broken in algorithms:
        print(f"{name:<10} {bits}-bit   {status:<15} {broken:<20}")

security_comparison()

### ✍️ Çalışma 3: SHA-2 (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Şəxsi heş:** SHA-256 və SHA-512 ilə öz adınızın heşini hesablayın.

2. **Performans müqayisəsi:** SHA-256 və SHA-512-nin performansını müqayisə edin. Hansı daha sürətlidir?

3. **Təhlükəsizlik səviyyəsi:** SHA-256-nın toqquşma dayanıqlılığı nədir? Bu rəqəmi şərh edin.

4. **Uzunluq genişləndirmə:** SHA-2-nin uzunluq genişləndirmə hücumuna qarşı həssas olmasının səbəbi nədir?

In [ ]:
# Çalışma 3 - Cavablar

print("📝 ÇALIŞMA 3 CAVABLARI")
print("=" * 80)

# 1. Şəxsi heş
print("\n1. ŞƏXSİ HEŞ (SHA-2):")
my_name = "Səməd Vəkilov"
h256 = hashlib.sha256(my_name.encode("utf-8")).hexdigest()
h512 = hashlib.sha512(my_name.encode("utf-8")).hexdigest()
print(f"   Ad: {my_name}")
print(f"   SHA-256: {h256}")
print(f"   SHA-512: {h512[:64]}...")

# 2. Performans müqayisəsi (artıq yuxarıda edildi)
print("\n2. PERFORMANS MÜQAYİSƏSİ:")
print("   Yuxarıdakı nəticələr platformadan asılıdır.")
print("   • Bəzi 64-bit sistemlərdə SHA-512 daha sürətli ola bilər")
print("   • Başqa sistemlərdə SHA-256 üstün ola bilər")
print("   • Qərar real ölçməyə əsasən verilməlidir")

# 3. Təhlükəsizlik səviyyəsi
print("\n3. TƏHLÜKƏSİZLİK SƏVİYYƏSİ:")
print("""
   SHA-256 toqquşma dayanıqlılığı: 2^128 ≈ 3.4 × 10^38
   • Bu, praktik olaraq qeyri-mümkün bir rəqəmdir
   • Dünyadakı bütün kompüterlərin birlikdə hesablama gücü ilə belə
     toqquşma tapmaq son dərəcə uzun vaxt tələb edər
   • SHA-256 bu gün təhlükəsiz hesab olunur
""")

# 4. Uzunluq genişləndirmə
print("\n4. UZUNLUQ GENİŞLƏNDİRMƏ:")
print("""
   SHA-2 uzunluq genişləndirmə hücumuna qarşı həssasdır, çünki:
   • Merkl-Damqord quruluşu istifadə edir
   • H(M) və len(M) məlumdursa, H(M || pad || X) hesablamaq olar
   • Bu səbəbdən H(key || message) əvəzinə HMAC istifadə edilməlidir

   Qorunma üsulları:
   • HMAC istifadə edin
   • SHA-3 (süngər quruluşu) bu hücuma qarşı davamlıdır
""")


## 🌐 Müasir tətbiqlərdə heş funksiyaları (15 dəq)

In [ ]:
def applications_demo():
    """
    Heş funksiyalarının müasir tətbiqləri
    """
    print("\n" + "=" * 70)
    print("🌐 HEŞ FUNKSİYALARININ TƏTBİQ SAHƏLƏRİ")
    print("=" * 70)

    print("""
    1. Kriptovalyutalar (Bitcoin):
       • Blok başlıqları və proof-of-work üçün ikili SHA-256
       • Ünvanlarda SHA-256 + RIPEMD-160 (HASH160)

    2. TLS/SSL sertifikatları:
       • SHA-256 (minimum tələb)
       • SHA-384, SHA-512 (yüksək təhlükəsizlik)

    3. SSH, IPsec:
       • SHA-256, SHA-384, SHA-512
       • HMAC-SHA-256 autentifikasiya üçün

    4. Git versiya idarəetmə:
       • SHA-1 ilə tarixi uyğunluq, SHA-256 dəstəyi də mövcuddur
       • Commit ID, obyekt identifikatorları

    5. Parol saxlanması:
       • Birbaşa SHA-256/512 istifadə etmək kifayət deyil
       • PBKDF2, bcrypt, scrypt və ya Argon2id daha uyğundur

    6. Fayl bütövlüyü:
       • SHA-256, SHA-512 (yükləmələr, yeniləmələr)
       • MD5 və SHA-1 yalnız köhnə uyğunluq hallarında görünə bilər
    """)

applications_demo()


### 8.1 Praktik tövsiyələr

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50;">
<h4>✅ Heş funksiyalarının seçimi üçün tövsiyələr</h4>
<ul>
    <li><b>MD5-dən qaçının:</b> Təhlükəsizlik tələb edən heç bir yerdə istifadə etməyin.</li>
    <li><b>SHA-1-i dayandırın:</b> Mövcud sistemlərdə SHA-1-i SHA-2 ilə əvəz edin.</li>
    <li><b>SHA-256 standart seçimdir:</b> Yeni sistemlər üçün əsas seçim.</li>
    <li><b>SHA-512 yüksək təhlükəsizlik üçün:</b> Daha yüksək təhlükəsizlik tələb edən sistemlərdə.</li>
    <li><b>HMAC istifadə edin:</b> Uzunluq genişləndirmə hücumundan qorunmaq üçün.</li>
    <li><b>Parol saxlanması:</b> PBKDF2, bcrypt, scrypt və ya Argon2 istifadə edin.</li>
</ul>
</div>

In [ ]:
def migration_strategy():
    """
    MD5/SHA-1-dən SHA-2-yə keçid strategiyası
    """
    print("\n" + "-" * 70)
    print("🔄 KEÇİD STRATEGİYASI")
    print("-" * 70)

    print("""
    MD5 və ya SHA-1 istifadə edən köhnə sistemlər üçün keçid planı:

    1. İnventarizasiya:
       • MD5/SHA-1 istifadə edən bütün yerləri müəyyənləşdirin
       • Kritiklik səviyyəsini təyin edin

    2. Dual hash (keçid dövrü):
       • Həm köhnə, həm də yeni heşi saxlayın
       • Köhnə sistemlərlə uyğunluğu qoruyun

    3. Yeni heşə keçid:
       • SHA-256-a keçin (minimum)
       • Mümkünsə SHA-384 və ya SHA-512

    4. Test:
       • Bütün funksionallığı test edin
       • Performansı ölçün

    5. Köhnə heşi ləğv edin:
       • Uyğunluq müddəti bitdikdən sonra köhnə heşi silin
    """)

migration_strategy()

### ✍️ Çalışma 4: Tətbiqlər (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Bitcoin heşi:** Bitcoin-də niyə ikili SHA-256 (SHA-256(SHA-256(x))) istifadə olunur?

2. **Git təhlükəsizliyi:** Git SHA-1 istifadə edir. Bu, təhlükəsizlik riski yaradırmı? Niyə?

3. **Seçim sualı:** Müasir yeni proqram sistemi qurursunuz. Hansı heş funksiyasını seçərdiniz?

4. **Keçid strategiyası:** MD5 istifadə edən köhnə bir sistemlə işləyirsiniz. Keçid strategiyası təklif edin.


In [ ]:
# Çalışma 4 - Cavablar

print("📝 ÇALIŞMA 4 CAVABLARI")
print("=" * 80)

# 1. Bitcoin ikili SHA-256
print("\n1. BİTCOİN İKİLİ SHA-256:")
sample = b"Bitcoin"
print(f"   Nümunə mətn: {sample!r}")
print(f"   SHA-256d: {double_sha256(sample)}")
print("""
   Bitcoin-də SHA-256d (SHA-256(SHA-256(x))) blok başlıqlarının heşlənməsində istifadə olunur.
   • Bu, protokolun dizayn seçimidir
   • İkiqat heşləmə sadə bir kompozisiya yaradır
   • Mining difficulty ayrıca hədəf dəyəri ilə təyin olunur;
     ikiqat heşləmə difficulty-ni öz-özünə "artırmır"
""")

# 2. Git təhlükəsizliyi
print("\n2. GIT TƏHLÜKƏSİZLİYİ:")
print("""
   Git-də SHA-1 istifadəsi risklidirmi?
   • Praktik toqquşmalar mövcud olsa da, Git konteksti hücumu çətinləşdirir
   • Git obyektlərində başlıq və struktur da nəzərə alınır
   • Buna baxmayaraq, Git SHA-256 dəstəyi əlavə edib
   • Yeni layihələr üçün SHA-256 daha münasib seçimdir
""")

# 3. Seçim sualı
print("\n3. MÜASİR SİSTEM ÜÇÜN HEŞ SEÇİMİ:")
print("""
   Yeni bir sistem üçün tövsiyələr:
   • Əsas seçim: SHA-256 (geniş dəstək, təhlükəsiz)
   • Yüksək təhlükəsizlik: SHA-512
   • Alternativ ailə: SHA-3
   • Parol saxlanması: Argon2id, scrypt, bcrypt və ya PBKDF2
""")

# 4. Keçid strategiyası (artıq yuxarıda verilib)
print("\n4. KEÇİD STRATEGİYASI:")
print("""
   MD5-dən SHA-256-a keçid üçün plan:
   1. MD5 istifadə edən bütün yerləri müəyyənləşdirin
   2. Dual hash dövrü: həm MD5, həm SHA-256 saxlayın
   3. Köhnə sistemlərlə uyğunluğu qoruyun
   4. SHA-256-a tam keçid edin
   5. MD5-in istifadəsini dayandırın

   Bu proses zamanı backward compatibility vacibdir!
""")


### ✅ Standart test vektorları ilə avtomatik yoxlama

Aşağıdakı hüceyrə MD5, SHA-1, SHA-224, SHA-256, SHA-384, SHA-512 və SHA-256d üçün tanınmış test vektorları ilə düzgünlüyü yoxlayır.


In [ ]:
def run_self_tests():
    """
    Standart test vektorları ilə düzgünlük yoxlaması.
    """
    print("\n" + "-" * 70)
    print("✅ STANDART TEST VEKTORLARI İLƏ YOXLAMA")
    print("-" * 70)

    tests = [
        ("MD5", hashlib.md5(b"abc").hexdigest(), "900150983cd24fb0d6963f7d28e17f72"),
        ("SHA-1", hashlib.sha1(b"abc").hexdigest(), "a9993e364706816aba3e25717850c26c9cd0d89d"),
        ("SHA-224", hashlib.sha224(b"abc").hexdigest(), "23097d223405d8228642a477bda255b32aadbce4bda0b3f7e36c9da7"),
        ("SHA-256", hashlib.sha256(b"abc").hexdigest(), "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"),
        ("SHA-384", hashlib.sha384(b"abc").hexdigest(), "cb00753f45a35e8bb5a03d699ac65007272c32ab0eded1631a8b605a43ff5bed8086072ba1e7cc2358baeca134c825a7"),
        ("SHA-512", hashlib.sha512(b"abc").hexdigest(), "ddaf35a193617abacc417349ae20413112e6fa4e89a97ea20a9eeee64b55d39a2192992a274fc1a836ba3c23a3feebbd454d4423643ce80e2a9ac94fa54ca49f"),
        ("SHA-256d", double_sha256(b"abc"), "4f8b42c22dd3729b519ba6f68d2da7cc5b2d606d05daed5ad5128cc03e6c6358"),
    ]

    all_ok = True
    print(f"{'Alqoritm':<10} {'Status':<8}")
    print("-" * 20)

    for name, got, expected in tests:
        ok = got == expected
        all_ok &= ok
        status = "OK" if ok else "XƏTA"
        print(f"{name:<10} {status:<8}")
        if not ok:
            print(f"  Gözlənən: {expected}")
            print(f"  Alınan  : {got}")

    if all_ok:
        print("\n🎉 Nəticə: bütün heş alqoritmləri düzgün implementasiya olunub.")
    else:
        print("\n❌ Bəzi implementasiyalarda problem var.")

    return all_ok

all_tests_ok = run_self_tests()


## 🖥️ İnteqrasiya edilmiş tətbiq (15 dəq)

In [ ]:
def hash_lab_menu():
    """
    Heş funksiyaları laboratoriyası interaktiv menyu
    """
    while True:
        print("\n" + "=" * 70)
        print("🔐 HEŞ FUNKSİYALARI LABORATORİYASI - Məşğələ 14")
        print("=" * 70)
        print("1. 📦 MD5 heşi hesabla")
        print("2. 📦 SHA-1 heşi hesabla")
        print("3. 📦 SHA-256 heşi hesabla")
        print("4. 📦 SHA-512 heşi hesabla")
        print("5. 📊 Bütün heşləri müqayisə et")
        print("6. ⚡ Performans testi")
        print("7. 💥 MD5 toqquşma nümunəsi")
        print("8. 📚 SHA-2 ailəsi haqqında məlumat")
        print("9. 💡 SHAttered hücumu haqqında")
        print("10. ₿ SHA-256d hesabla")
        print("0. ❌ Çıxış")
        print("=" * 70)

        choice = input("📌 Seçiminiz: ")

        if choice == '1':
            text = input("📨 Mətn daxil edin: ")
            print(f"\n🔹 MD5: {hashlib.md5(text.encode('utf-8')).hexdigest()}")

        elif choice == '2':
            text = input("📨 Mətn daxil edin: ")
            print(f"\n🔹 SHA-1: {hashlib.sha1(text.encode('utf-8')).hexdigest()}")

        elif choice == '3':
            text = input("📨 Mətn daxil edin: ")
            print(f"\n🔹 SHA-256: {hashlib.sha256(text.encode('utf-8')).hexdigest()}")

        elif choice == '4':
            text = input("📨 Mətn daxil edin: ")
            print(f"\n🔹 SHA-512: {hashlib.sha512(text.encode('utf-8')).hexdigest()}")

        elif choice == '5':
            text = input("📨 Mətn daxil edin: ").encode("utf-8")
            print(f"\nMD5:      {hashlib.md5(text).hexdigest()}")
            print(f"SHA-1:    {hashlib.sha1(text).hexdigest()}")
            print(f"SHA-256:  {hashlib.sha256(text).hexdigest()}")
            print(f"SHA-512:  {hashlib.sha512(text).hexdigest()}")
            print(f"SHA-256d: {double_sha256(text)}")

        elif choice == '6':
            try:
                size = int(input("Test məlumatının ölçüsü (KB): ") or "1024")
                if size <= 0:
                    raise ValueError
            except ValueError:
                print("❌ Ölçü müsbət tam ədəd olmalıdır.")
                continue

            data = os.urandom(size * 1024)
            repeats = 200 if size <= 10 else 50 if size <= 100 else 10

            t256 = mean_hash_time(hashlib.sha256, data, repeats)
            t512 = mean_hash_time(hashlib.sha512, data, repeats)

            print(f"\n{size} KB məlumat üçün (orta {repeats} təkrar):")
            print(f"SHA-256: {t256:.4f} ms")
            print(f"SHA-512: {t512:.4f} ms")

        elif choice == '7':
            md5_collision_demo()

        elif choice == '8':
            print("\n" + "=" * 60)
            print("📚 SHA-2 AİLƏSİ")
            print("=" * 60)
            print("""
            SHA-224: 224-bit çıxış (32-bit sözlər, 64 raund)
            SHA-256: 256-bit çıxış (32-bit sözlər, 64 raund)
            SHA-384: 384-bit çıxış (64-bit sözlər, 80 raund)
            SHA-512: 512-bit çıxış (64-bit sözlər, 80 raund)

            Təhlükəsizlik:
            • SHA-256: 128-bit təhlükəsizlik
            • SHA-512: 256-bit təhlükəsizlik

            Tətbiqlər:
            • Bitcoin blok başlıqları: SHA-256d
            • TLS 1.3: SHA-256, SHA-384
            • SSH: SHA-256, SHA-512
            • GPG: SHA-256, SHA-512
            """)

        elif choice == '9':
            shattered_demo()

        elif choice == '10':
            text = input("📨 Mətn daxil edin: ")
            print(f"\n🔹 SHA-256d: {double_sha256(text)}")

        elif choice == '0':
            print("👋 Proqram bitdi. Sağ olun!")
            break

        else:
            print("❌ Yanlış seçim")

# Proqramı işə sal (istəyə bağlı)
# hash_lab_menu()


## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: Heş funksiyaları paketi (3 bal)

Aşağıdakı funksiyaları özündə birləşdirən Python paketi yazın:

```
hash_package/
├── __init__.py
├── md5_demo.py          # MD5 heşləri, toqquşma nümunəsi
├── sha1_demo.py         # SHA-1 heşləri, SHAttered məlumatı
├── sha2_family.py       # SHA-2 ailəsinin bütün variantları
├── performance.py       # SHA-256 vs SHA-512 performans müqayisəsi
├── security_analysis.py # Təhlükəsizlik xüsusiyyətlərinin müqayisəsi
└── main.py              # Bütün funksiyaları birləşdirən interaktiv menyu
```

**Tələblər:**
* Hər bir funksiya üçün docstring yazın
* Səhv hallarını idarə edin (try-except)
* Kod təmiz və oxunaqlı olmalıdır
* Hər modul üçün test nümunələri əlavə edin
* Qrafik funksiyaları matplotlib varsa işləsin

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **Fayl heşləri:** 1 MB ölçülü təsadüfi fayl yaradın və onun MD5, SHA-1, SHA-256 heşlərini hesablayın.

2. **Performans qrafiki:** SHA-256 və SHA-512-nin performansını müxtəlif ölçülü fayllar (1KB, 10KB, 100KB, 1MB, 10MB) üçün müqayisə edin. Qrafik çəkin.

3. **Kiçik toqquşma:** MD5 üçün toqquşma axtarışı aparan proqram yazın (çox kiçik heş ölçüsü ilə, məsələn 16-bit). Nə qədər sınaqdan sonra toqquşma tapıldı?

4. **Uzunluq genişləndirmə:** SHA-256 üçün `tag = SHA256(secret || message)` formasında qurulmuş sadə MAC-a qarşı uzunluq genişləndirmə hücumunu nümayiş etdirin. Python-un standart `hashlib` modulu daxili SHA-256 vəziyyətini davam etdirməyə imkan vermir; buna görə bu tapşırıqda ya müəllimin təsdiqlədiyi üçüncü-tərəf kitabxanasından/CLI alətindən istifadə edin, ya da SHA-256 kompressiya vəziyyətini təyin etməyə imkan verən sadə implementasiya yazın. Minimal yoxlama: seçilmiş gizli `secret`, ilkin `message`, `tag = SHA256(secret || message)` və əlavə ediləcək `suffix` üçün hücum nəticəsində `forged_message = message || pad(secret || message) || suffix` və `forged_tag` alınmalıdır; yoxlama zamanı `forged_tag == SHA256(secret || forged_message)` olmalıdır. Hücum zamanı `secret`-in məzmunundan istifadə etməyin, yalnız onun uzunluğunu təxmin edin.

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və aşağıdakı suallara cavab tapın. Cavablarınızı 1-2 səhifəlik hesabat şəklində təqdim edin:

1. **Ron Rivest:** Ron Rivest kimdir və onun kriptoqrafiyaya töhfələri nələrdir? (RSA, MD2, MD4, MD5, RC4)

2. **Xiaoyun Wang:** Xiaoyun Wang-ın MD5 və SHA-1-ə qarşı hücumları nə dərəcədə əhəmiyyətli idi? Onun metodları nə idi?

3. **SHAttered:** SHAttered hücumunun texniki detalları nələrdir? $2^{63}$ mürəkkəblik nə deməkdir və praktik olaraq necə reallaşdırıldı?

4. **SHA-3 müsabiqəsi:** NIST niyə SHA-3 müsabiqəsini təşkil etdi? SHA-2-nin bilinən problemləri varmı? Keccak niyə seçildi?

**Format tələbləri:**
* PDF formatında təqdim edin
* Ən azı 5 qaynaq göstərin (kitab, məqalə, veb səhifə)
* Öz sözlərinizlə yazın (copy-paste yox)
* Mümkünsə, qrafiklər və ya diaqramlar əlavə edin

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<p>Bu məşğələdə öyrəndiklərimiz:</p>
<ul>
    <li>✅ <b>MD5:</b> 128-bit, 1991, Ron Rivest. 2004-cü ildə toqquşma tapıldı. Bu gün tamamilə sındırılmışdır.</li>
    <li>✅ <b>SHA-1:</b> 160-bit, 1995, NSA. 2017-ci ildə SHAttered hücumu ilə praktik toqquşma tapıldı. Etibarsızdır.</li>
    <li>✅ <b>SHA-256:</b> 256-bit, 2001, NSA. 128-bit təhlükəsizlik səviyyəsi. Hal-hazırda təhlükəsizdir.</li>
    <li>✅ <b>SHA-512:</b> 512-bit, 2001, NSA. 256-bit təhlükəsizlik səviyyəsi. Hal-hazırda təhlükəsizdir.</li>
    <li>✅ <b>Təhlükəsizlik müqayisəsi:</b> MD5 < SHA-1 < SHA-256 < SHA-512</li>
</ul>
<p><i>Praktik tövsiyə: Yeni sistemlər üçün SHA-256 standart seçimdir. Daha yüksək təhlükəsizlik tələb olunduqda SHA-512 istifadə edin. MD5 və SHA-1-dən qaçının.</i></p>
</div>

### 💭 Müzakirə sualları

1. Bu məşğələdə ən maraqlı tapdığınız nə oldu?
2. Hansı heş funksiyası sizə daha məntiqli gəlir? Niyə?
3. MD5 və SHA-1-in sındırılmasının praktik nəticələri nələrdir?
4. SHA-256 nə qədər müddət təhlükəsiz qalacaq? Nə vaxta qədər istifadə edə bilərik?
5. SHA-3 (Keccak) SHA-2-dən nə ilə fərqlənir? (növbəti məşğələnin mövzusu)
6. Parol saxlanmasında niyə təkbaşına SHA-256 istifadə etmək olmaz?
7. Kvant kompüterlər SHA-2-yə təhlükə yaradırmı?

## 📚 Əlavə resurslar

* 📘 **hashlib sənədləşməsi:** [https://docs.python.org/3/library/hashlib.html](https://docs.python.org/3/library/hashlib.html)
* 📙 **RFC 1321 (MD5):** [https://tools.ietf.org/html/rfc1321](https://tools.ietf.org/html/rfc1321)
* 📗 **RFC 3174 (SHA-1):** [https://tools.ietf.org/html/rfc3174](https://tools.ietf.org/html/rfc3174)
* 📕 **FIPS 180-4 (SHA-2):** [https://csrc.nist.gov/publications/detail/fips/180/4/final](https://csrc.nist.gov/publications/detail/fips/180/4/final)
* 📘 **SHAttered hücumu:** [https://shattered.io/](https://shattered.io/)
* 📙 **Wang, X. (2005). "How to Break MD5 and Other Hash Functions"**
* 📗 **Google Online Security Blog - SHAttered:** [https://security.googleblog.com/2017/02/announcing-first-sha1-collision.html](https://security.googleblog.com/2017/02/announcing-first-sha1-collision.html)
* 📘 **NIST SP 800-107 Rev. 1 (tarixi/geri çəkilmiş heş funksiyaları tövsiyələri):** [https://csrc.nist.gov/pubs/sp/800/107/r1/final](https://csrc.nist.gov/pubs/sp/800/107/r1/final)
* 📕 **NIST heş funksiyaları üzrə cari siyasət:** [https://csrc.nist.gov/Projects/hash-functions/nist-policy-on-hash-functions](https://csrc.nist.gov/Projects/hash-functions/nist-policy-on-hash-functions)
* 📕 **NIST SP 800-131A və SP 800-57 (alqoritm/key uzunluğu keçidləri və təhlükəsizlik gücü üçün cari rəhbərlik):** [https://csrc.nist.gov/publications/detail/sp/800-131a/rev-2/final](https://csrc.nist.gov/publications/detail/sp/800-131a/rev-2/final), [https://csrc.nist.gov/publications/detail/sp/800-57-part-1/rev-5/final](https://csrc.nist.gov/publications/detail/sp/800-57-part-1/rev-5/final)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 14 tamamlandı!</h2>
<p>Bütün kodları və tapşırıq cavablarını növbəti məşğələyə qədər təqdim edin.</p>
<p><em>Kodlar aydın şərhlərlə yazılmalı və asan oxunmalıdır. Hər bir funksiyanın nə etdiyini izah edən şərhlər əlavə edin.</em></p>
<p style="font-size: 1.2em; margin-top: 15px;">🔐 <b>Heş funksiyaları - MD5-dən SHA-2-yə təkamül!</b></p>
</div>